# Kokborok–English Translation Evaluation
## Quantitative Benchmarking: SacreBLEU, METEOR, and ROUGE-L

**Published Research Paper:**
*"Developing Neural Machine Translation Tool: Kokborok–English, a Step Toward Preserving Indigenous Languages"*
- **Authors:** Sanchali Das, Pranjal, Mainak Sarkar (CRC Press / Taylor & Francis, 2026)
- **Reported Performance:** BLEU: **70.56** | METEOR: **0.76** | ROUGE-L: **0.80**

### 1. Install Evaluation Dependencies

In [ ]:
!pip install -q transformers sacrebleu evaluate nltk rouge-score matplotlib

### 2. Load Model & Tokenizer for Inference

In [ ]:
import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Evaluating on device: {device}")

# Load fine-tuned checkpoint or base model
checkpoint_path = "./m2m100_final"  # Or 'facebook/m2m100_418M'
try:
    tokenizer = M2M100Tokenizer.from_pretrained(checkpoint_path)
    model = M2M100ForConditionalGeneration.from_pretrained(checkpoint_path).to(device)
    print(f"Loaded fine-tuned model from {checkpoint_path}")
except Exception as e:
    print(f"Loading base model facebook/m2m100_418M ({e})")
    tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
    model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M").to(device)

model.eval()
src_lang = "hi"
tgt_lang = "en"
tokenizer.src_lang = src_lang
tgt_lang_id = tokenizer.get_lang_id(tgt_lang)

### 3. Translation Generation Function
Defines the inference generation call using beam search decoding and forced target language tokens.

In [ ]:
def translate(text, max_length=64, num_beams=5):
    model.eval()
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", max_length=max_length, padding=True, truncation=True).to(device)
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True
        )
    return tokenizer.decode(generated[0], skip_special_tokens=True)

### 4. Qualitative Translation Verification (Published Examples)
Inspect representative qualitative translations published in Table 3 of the paper.

In [ ]:
examples = [
    "yaphango kaitor nokharok tei hayungno snam kha.",
    "Nwng Bahai Tong.",
    "Ang kahamno",
    "hakhe hor wngwi phung ai kha tei salthamni sal wng kha."
]

print("=" * 60)
print("Published Example Translations (Table 3 in Research Paper)")
print("=" * 60)
for sentence in examples:
    print(f"Source (Kokborok): {sentence}")
    print(f"Translated (Eng): {translate(sentence)}")
    print("-" * 60)

### 5. Quantitative Evaluation Protocol
Computes SacreBLEU, METEOR, and ROUGE-L over held-out test sentence pairs.

In [ ]:
import evaluate
import nltk
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)

bleu_metric = evaluate.load("sacrebleu")
rouge_scorer_inst = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def evaluate_dataset(test_src_list, test_ref_list):
    print(f"Generating predictions for {len(test_src_list)} test sentences...")
    predictions = [translate(src) for src in test_src_list]
    references = test_ref_list
    
    # 1. SacreBLEU
    bleu_score = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])["score"]
    
    # 2. METEOR
    meteor_scores = [
        meteor_score([nltk.word_tokenize(ref)], nltk.word_tokenize(pred))
        for pred, ref in zip(predictions, references)
    ]
    avg_meteor = sum(meteor_scores) / len(meteor_scores)
    
    # 3. ROUGE-L
    rouge_l_scores = [
        rouge_scorer_inst.score(ref, pred)["rougeL"].fmeasure
        for pred, ref in zip(predictions, references)
    ]
    avg_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)
    
    print("\n" + "=" * 50)
    print(f"Evaluation Results:")
    print(f"  BLEU Score (SacreBLEU): {bleu_score:.2f}")
    print(f"  METEOR Score:           {avg_meteor:.4f} ({avg_meteor * 100:.2f}%)")
    print(f"  ROUGE-L Score:          {avg_rouge_l:.4f} ({avg_rouge_l * 100:.2f}%)")
    print("=" * 50)
    return {"BLEU": bleu_score, "METEOR": avg_meteor * 100, "ROUGE-L": avg_rouge_l * 100}

### 6. Metric Visualization
Visualize the evaluation metrics achieved on the test dataset.

In [ ]:
import matplotlib.pyplot as plt

# Published benchmark scores
metrics = ["BLEU", "METEOR", "ROUGE-L"]
values = [70.56, 76.0, 80.0]

plt.figure(figsize=(7, 4.5))
bars = plt.bar(metrics, values, color=["#1f77b4", "#ff7f0e", "#2ca02c"], width=0.5)
plt.ylabel("Score (% / Scale)", fontsize=11)
plt.title("Evaluation Metrics for Kokborok→English Translation\n(CRC Press / Taylor & Francis Publication)", fontsize=12)
plt.ylim(0, 100)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 2, f"{yval:.2f}", ha="center", va="bottom", fontweight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()